### Analysis on the effect of IMO 2020, and how new ships should be fueled

Data Collection + Cleaning

In [3]:
import pandas as pd
import os  #for env
from dotenv import load_dotenv
from fredapi import Fred  #crude oil price data

load_dotenv()
import matplotlib.pyplot as plt

In [4]:
#annual breakdown
annual_data = pd.read_csv("../data/raw/BunkerSalesBreakdownAnnual.csv")
#monthly breakdown
monthly_data = pd.read_csv("../data/raw/BunkerSalesBreakdownMonthly.csv")

Basic Information

In [5]:
annual_data.info()
annual_data.describe()

<class 'pandas.DataFrame'>
RangeIndex: 496 entries, 0 to 495
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   year          496 non-null    int64  
 1   bunker_type   496 non-null    str    
 2   bunker_sales  496 non-null    float64
dtypes: float64(1), int64(1), str(1)
memory usage: 11.8 KB


,year,bunker_sales
count,496.000000,496.000000
mean,2010.000000,2244.508004
std,8.953302,7922.469642
min,1995.000000,0.000000
25%,2002.000000,0.000000
50%,2010.000000,0.000000
75%,2018.000000,2.542500
max,2025.000000,48469.470000


In [6]:
monthly_data.info()
monthly_data.describe()

<class 'pandas.DataFrame'>
RangeIndex: 5968 entries, 0 to 5967
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   month         5968 non-null   str    
 1   bunker_type   5968 non-null   str    
 2   bunker_sales  5968 non-null   float64
dtypes: float64(1), str(2)
memory usage: 140.0 KB


,bunker_sales
count,5968.000000
mean,187.416860
std,662.921049
min,0.000000
25%,0.000000
50%,0.000000
75%,0.040000
max,4395.900000


In [7]:
annual_data.shape

(496, 3)

In [8]:
monthly_data.shape

(5968, 3)

In [9]:
sorted(annual_data["bunker_type"].unique())

['Ammonia',
 'B100',
 'Bio-Blended Low Sulphur Fuel Oil',
 'Bio-Blended Low Sulphur Marine Gas Oil',
 'Bio-Blended Marine Diesel Oil',
 'Bio-Blended Marine Fuel Oil',
 'Bio-Blended Marine Gas Oil',
 'Bio-Blended Ultra Low Sulphur Fuel Oil',
 'LNG',
 'Low Sulphur Fuel Oil',
 'Low Sulphur Marine Gas Oil',
 'Marine Diesel Oil',
 'Marine Fuel Oil',
 'Marine Gas Oil',
 'Methanol',
 'Ultra Low Sulphur Fuel Oil']

Simplifying the data columns

Despite the 16 unique types of fuel types present in the dataset, some types can be grouped together under the same categories.
These include:
   - **MFO** (Marine Fuel Oil + Bio-Blended MFO)
   - **LSFO** (Low Sulphur Fuel Oil + Bio-Blended LSFO)
   - **ULSFO** (Ultra Low Sulphur Fuel Oil + Bio-Blended ULSFO)
   - **MGO** (Marine Gas Oil + Bio-Blended MGO)
   - **LSMGO** (Low Sulphur Marine Gas Oil + Bio-Blended LSMGO)
   - **MDO** (Marine Diesel Oil + Bio-Blended MDO)
   - **LNG**
   - **Alternative** (Methanol + Ammonia + B100)

The new "Simplified" Table will now have 8 categories instead

In [10]:
category_map = {

    #MFO
    'Marine Fuel Oil' : 'MFO',
    'Bio-Blended Marine Fuel Oil' : 'MFO',

    #LSFO
    'Low Sulphur Fuel Oil' : 'LSFO',
    'Bio-Blended Low Sulphur Fuel Oil' : 'LSFO',

    #ULSFO
    'Ultra Low Sulphur Fuel Oil' : 'ULSFO',
    'Bio-Blended Ultra Low Sulphur Fuel Oil' : 'ULSFO',

    #MGO
    'Marine Gas Oil': 'MGO',
    'Bio-Blended Marine Gas Oil' : 'MGO',

    #LSMGO
    'Low Sulphur Marine Gas Oil' : 'LSMGO',
    'Bio-Blended Low Sulphur Marine Gas Oil' : 'LSMGO',

    #MDO
    'Marine Diesel Oil': 'MDO',
    'Bio-Blended Marine Diesel Oil' : 'MDO',

    #LNG and alternatives
    'LNG': 'LNG',
    'Bio-Blended LNG': 'LNG',
    'Ammonia': 'Alternative',
    'Methanol': 'Alternative',
    'B100' : 'Alternative',
}

In [11]:
#Making sure that all categories are accounted for
set(annual_data['bunker_type'].unique()) - set(category_map.keys())

set()

Grouping the similar categories together

In [12]:
annual_data['fuel_category'] = annual_data['bunker_type'].map(category_map)
annual_data['fuel_category']

0              MDO
1              MDO
2              MGO
3              MGO
4            LSMGO
          ...     
491          ULSFO
492    Alternative
493            LNG
494    Alternative
495    Alternative
Name: fuel_category, Length: 496, dtype: str

In [17]:
annual_data

,year,bunker_type,bunker_sales,fuel_category
0,1995,Marine Diesel Oil,700.10,MDO
1,1995,Bio-Blended Marine Diesel Oil,0.00,MDO
2,1995,Marine Gas Oil,1317.30,MGO
3,1995,Bio-Blended Marine Gas Oil,0.00,MGO
4,1995,Low Sulphur Marine Gas Oil,0.00,LSMGO
...,...,...,...,...
491,2025,Bio-Blended Ultra Low Sulphur Fuel Oil,0.71,ULSFO
492,2025,B100,25.50,Alternative
493,2025,LNG,571.35,LNG
494,2025,Methanol,3.01,Alternative


In [16]:
simplified_ad_long = annual_data.groupby(['year', 'fuel_category'])['bunker_sales'].sum().reset_index()
simplified_ad_long.head()

,year,fuel_category,bunker_sales
0,1995,Alternative,0.0
1,1995,LNG,0.0
2,1995,LSFO,0.0
3,1995,LSMGO,0.0
4,1995,MDO,700.1


In [ ]:
#CSV creation

In [18]:
simplified_ad_long.to_csv('../data/cleaned/BunkerByCategoryLong.csv')

Pivoting to a wide format for easier plotting

In [24]:
simplified_ad_wide = annual_data.groupby(['year', 'fuel_category'])['bunker_sales'].sum().unstack()
simplified_ad_wide

fuel_category,Alternative,LNG,LSFO,LSMGO,MDO,MFO,MGO,ULSFO
year,,,,,,,,
1995,0.00,0.00,0.00,0.00,700.10,15471.80,1317.30,0.00
1996,0.00,0.00,0.00,0.00,556.91,15174.88,1206.69,0.00
1997,0.00,0.00,0.00,0.00,480.50,15229.28,1230.97,0.00
1998,0.00,0.00,0.00,0.00,419.56,16283.79,1360.47,0.00
1999,0.00,0.00,0.00,0.00,400.20,17148.69,1342.31,0.00
2000,0.00,0.00,0.00,0.00,296.66,17096.37,1257.76,0.00
2001,0.00,0.00,0.00,0.00,249.80,18426.52,1675.35,0.00
2002,0.00,0.00,0.00,0.00,187.07,18156.45,1752.86,0.00
2003,0.00,0.00,0.00,0.00,166.20,19307.70,1335.07,0.00


In [25]:
simplified_ad_wide.to_csv('../data/cleaned/BunkerByCategoryWide.csv')

After IMO 2020, a stricter limit of sulphur content was placed on all fuel, reducing the limit to 0.5% from 3.5%

This was done as an effort to help curb air pollution and environmental damage.
As such, ships and companies that were initially burning fuel that fell within the previous limit of 3.5% but not the new limit of 0.5% had
2 options going forward:
1. Switch to compliant fuel (More expensive but no ship modifications needed)
2. Install a marine scrubber (Filters sulphur from the exhaust but use the same fuel)
3. Switch to alternative fuel (Requires new engines but is future-proof)

Before IMO 2020, Certain regions (Baltic Sea, North Sea, US & Canadian coasts, US Caribbean) already had stricter sulphur regulation limits at 0.1%, fuels like Ultra low sulphur fuel oil would fit into this category of ECA(Emission control area) compliant fuels



In [26]:
compliance_map = {
      'MFO': 'Non-Compliant',
      'LSFO': 'IMO 2020 Compliant',
      'MDO': 'IMO 2020 Compliant',
      'MGO': 'IMO 2020 Compliant',
      'ULSFO': 'ECA Compliant',
      'LSMGO': 'ECA Compliant',
      'LNG': 'Alternative',
      'Alternative': 'Alternative',
  }

In [29]:
annual_data['compliance_tier'] = annual_data['fuel_category'].map(compliance_map)
annual_data['compliance_tier']

0      IMO 2020 Compliant
1      IMO 2020 Compliant
2      IMO 2020 Compliant
3      IMO 2020 Compliant
4           ECA Compliant
              ...        
491         ECA Compliant
492           Alternative
493           Alternative
494           Alternative
495           Alternative
Name: compliance_tier, Length: 496, dtype: str

In [32]:
by_compliance = annual_data.groupby(['year', 'compliance_tier'])['bunker_sales'].sum().unstack()
by_compliance

compliance_tier,Alternative,ECA Compliant,IMO 2020 Compliant,Non-Compliant
year,,,,
1995,0.00,0.00,2017.40,15471.80
1996,0.00,0.00,1763.60,15174.88
1997,0.00,0.00,1711.47,15229.28
1998,0.00,0.00,1780.03,16283.79
1999,0.00,0.00,1742.51,17148.69
2000,0.00,0.00,1554.42,17096.37
2001,0.00,0.00,1925.15,18426.52
2002,0.00,0.00,1939.93,18156.45
2003,0.00,0.00,1501.27,19307.70


In [33]:
by_compliance.to_csv('../data/cleaned/AnnualBunkerByCompliance.csv')

Since the data will be affected by the raw price of crude oil, we can use the crude oil indexes to better isolate the effect of imo 2020 on the growth of the alternative and compliant products.

There are two main indexes used for determining the price of crude oil,

Brent, which is the UK's index
WTI, which is the US's index

In [36]:
fred = Fred(api_key=os.getenv('FRED_API_KEY'))
brent = fred.get_series('DCOILBRENTEU', observation_start='1998-01-01')
brent

1998-01-01      NaN
1998-01-02    15.77
1998-01-05    15.29
1998-01-06    15.48
1998-01-07    15.33
              ...  
2026-02-24    71.21
2026-02-25    70.69
2026-02-26    71.66
2026-02-27    71.32
2026-03-02    77.24
Length: 7348, dtype: float64

In [40]:
wti = fred.get_series('DCOILWTICO', observation_start='1998-01-01')
wti

1998-01-01      NaN
1998-01-02    17.41
1998-01-05    16.95
1998-01-06    16.64
1998-01-07    16.91
              ...  
2026-02-24    65.62
2026-02-25    65.30
2026-02-26    65.10
2026-02-27    66.96
2026-03-02    71.13
Length: 7348, dtype: float64